# Risk Engine — Adaptive Road Vision

Combine the outputs of the previous pipeline stages into a transparent, rule-based road-risk score.

**Inputs:** depth-object estimates, per-frame tracking diagnostics, selected-detector metadata, and condition labels. Paths are checked before analysis; synthetic demo rows are used only when `USE_DEMO_DATA = True` is explicitly selected.

**Signals:** relative proximity within each image, detection confidence, frame-level tracking persistence, scene condition, and configurable class weights.

> This is a project-level perception score, **not** a collision probability or a safety certification. Monocular depth is relative unless independently calibrated.


In [ ]:
from pathlib import Path
import os, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. Update PROJECT_ROOT to your local project folder."
    )
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "risk.yaml"
DEPTH_PATH = PROJECT_ROOT / "results" / "depth" / "depth_object_estimates.csv"
TRACK_PATH = PROJECT_ROOT / "results" / "tracking" / "bytetrack_per_frame.csv"
SELECTION_PATH = PROJECT_ROOT / "results" / "detection" / "final_detector_selection.json"
RESULT_ROOT = PROJECT_ROOT / "results" / "risk"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# Keep False for real project evaluation. Set True only to test the formulas with synthetic rows.
USE_DEMO_DATA = False

print("Project root:", PROJECT_ROOT)
print("Risk config:", CONFIG_PATH)
print("Depth input:", DEPTH_PATH)
print("Tracking input:", TRACK_PATH)


## 1. Load project risk configuration

If `configs/risk.yaml` contains project-specific parameters, they are used. Missing parameters fall back to explicit notebook defaults.


In [ ]:
import yaml

risk_cfg = {}

if CONFIG_PATH.exists():
    try:
        risk_cfg = yaml.safe_load(
            CONFIG_PATH.read_text(encoding="utf-8")
        ) or {}
        print("Loaded risk configuration.")
        print(risk_cfg)
    except Exception as e:
        print("Could not parse risk.yaml:", e)
else:
    print("risk.yaml not found; using notebook defaults.")


## 2. Risk-engine parameters

The defaults are intentionally transparent so every contribution can be inspected and changed.


In [ ]:
DEFAULTS = {
    "near_distance_threshold": 0.25,
    "critical_distance_threshold": 0.60,
    "confidence_floor": 0.25,
    "persistence_reference_frames": 10,

    "condition_factor": {
        "Day": 1.00,
        "Night": 1.20,
        "Rain": 1.15,
        "Fog": 1.25,
        "Unknown": 1.05,
    },

    "class_weight": {
        0: 1.30,  # pedestrian
        1: 1.25,  # rider
        2: 1.00,  # car
        3: 1.05,  # truck
        4: 1.05,  # bus
        5: 1.00,  # train
        6: 1.20,  # motorcycle
        7: 1.20,  # bicycle
    },

    "level_thresholds": {
        "LOW": 0.25,
        "MEDIUM": 0.50,
        "HIGH": 0.75,
    },
}

# Merge only keys explicitly present in the project config.
cfg = risk_cfg

near_distance_threshold = float(
    cfg.get("near_distance_threshold", DEFAULTS["near_distance_threshold"])
)
critical_distance_threshold = float(
    cfg.get("critical_distance_threshold", DEFAULTS["critical_distance_threshold"])
)
confidence_floor = float(
    cfg.get("confidence_floor", DEFAULTS["confidence_floor"])
)
persistence_reference_frames = int(
    cfg.get(
        "persistence_reference_frames",
        DEFAULTS["persistence_reference_frames"]
    )
)

condition_factor = DEFAULTS["condition_factor"].copy()
condition_factor.update(cfg.get("condition_factor", {}) or {})

class_weight = DEFAULTS["class_weight"].copy()
for k, v in (cfg.get("class_weight", {}) or {}).items():
    class_weight[int(k)] = float(v)

level_thresholds = DEFAULTS["level_thresholds"].copy()
level_thresholds.update(cfg.get("level_thresholds", {}) or {})

print("Condition factors:", condition_factor)
print("Class weights:", class_weight)
print("Level thresholds:", level_thresholds)


## 3. Load depth, tracking, and detector-selection artifacts

In [ ]:
# Load real artifacts. Demo rows are never silently mixed with real data.
missing = [str(p) for p in (DEPTH_PATH, TRACK_PATH) if not p.is_file()]
if missing and not USE_DEMO_DATA:
    raise FileNotFoundError(
        "Required input file(s) missing:\n- " + "\n- ".join(missing) +
        "\nRun the depth and tracking notebooks first, or set USE_DEMO_DATA=True only for a formula smoke-test."
    )

if USE_DEMO_DATA and missing:
    print("WARNING: Using synthetic demonstration rows; do not report these as project results.")
    depth_df = pd.DataFrame([
        {"image_name": "demo_001.jpg", "condition": "Night", "class_id": 0, "confidence": 0.85, "relative_depth": 0.35, "detection_index": 0},
        {"image_name": "demo_001.jpg", "condition": "Night", "class_id": 2, "confidence": 0.90, "relative_depth": 0.20, "detection_index": 1},
        {"image_name": "demo_002.jpg", "condition": "Rain", "class_id": 1, "confidence": 0.78, "relative_depth": 0.50, "detection_index": 0},
    ])
    tracking_df = pd.DataFrame([
        {"image_name": "demo_001.jpg", "sequence_id": "seq_01", "frame_number": 1, "active_tracks": 2, "track_ids": "1,2", "mode": "original"},
        {"image_name": "demo_002.jpg", "sequence_id": "seq_01", "frame_number": 2, "active_tracks": 1, "track_ids": "3", "mode": "original"},
    ])
else:
    depth_df = pd.read_csv(DEPTH_PATH)
    tracking_df = pd.read_csv(TRACK_PATH)

selection = {}
if SELECTION_PATH.is_file():
    try:
        selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as exc:
        print(f"WARNING: Could not read detector-selection JSON: {exc}")

# Validate essential schema and fill optional metadata consistently.
required_depth = {"image_name", "class_id", "confidence", "relative_depth"}
missing_cols = required_depth - set(depth_df.columns)
if missing_cols:
    raise ValueError(f"Depth CSV is missing required columns: {sorted(missing_cols)}")
if "condition" not in depth_df.columns:
    depth_df["condition"] = "Unknown"
if "image_id" not in depth_df.columns:
    depth_df["image_id"] = depth_df["image_name"]
if "mode" not in tracking_df.columns:
    tracking_df["mode"] = "original"
for col, default in [("sequence_id", ""), ("track_ids", ""), ("active_tracks", 0), ("detections", 0), ("mean_confidence", 0.0)]:
    if col not in tracking_df.columns:
        tracking_df[col] = default

print("Depth rows:", len(depth_df))
print("Tracking rows:", len(tracking_df))
print("Selected detector:", selection.get("selected_detector", "not recorded"))
display(depth_df.head())
display(tracking_df.head())


## 4. Prepare tracking persistence information

The tracking file stores per-frame track IDs. We count appearances by `(mode, sequence_id, track_id)`. This is a **frame-level proxy**: without detection-to-track associations, persistence cannot be assigned to each depth object individually.


In [ ]:
def parse_track_ids(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [
        int(x) for x in str(value).split(",")
        if str(x).strip()
    ]


track_occurrences = {}

for row in tracking_df.itertuples(index=False):
    ids = parse_track_ids(getattr(row, "track_ids", ""))
    sequence_id = str(getattr(row, "sequence_id", ""))
    mode = str(getattr(row, "mode", "original"))

    for tid in ids:
        key = (mode, sequence_id, tid)
        track_occurrences[key] = track_occurrences.get(key, 0) + 1

print("Unique track keys:", len(track_occurrences))


## 5. Relative-depth risk transform

Depth Anything provides relative depth. The engine therefore normalizes depth **within each image** rather than claiming meters.

Higher normalized proximity means the object is relatively closer within that image.


In [ ]:
def image_depth_proximity(group):
    values = pd.to_numeric(group["relative_depth"], errors="coerce").to_numpy(dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros(len(values), dtype=float)
    lo, hi = np.nanpercentile(values[finite], [5, 95])
    if hi <= lo:
        # Equal/near-equal estimates do not provide an ordering within this frame.
        result = np.full(len(values), 0.5, dtype=float)
        result[~finite] = 0.0
        return result
    result = np.zeros(len(values), dtype=float)
    result[finite] = np.clip((values[finite] - lo) / (hi - lo), 0, 1)
    return result

# Assign by original row index to avoid pandas groupby.apply index-alignment differences.
depth_df = depth_df.copy()
depth_df["relative_depth"] = pd.to_numeric(depth_df["relative_depth"], errors="coerce")
depth_df["confidence"] = pd.to_numeric(depth_df["confidence"], errors="coerce").fillna(0).clip(0, 1)
depth_df["class_id"] = pd.to_numeric(depth_df["class_id"], errors="coerce")
if depth_df["class_id"].isna().any():
    raise ValueError("Depth CSV contains missing or non-numeric class_id values.")
depth_df["class_id"] = depth_df["class_id"].astype(int)
depth_df["proximity"] = 0.0
for _, idx in depth_df.groupby("image_name", sort=False).groups.items():
    depth_df.loc[idx, "proximity"] = image_depth_proximity(depth_df.loc[idx])

display(depth_df[["image_name", "condition", "class_id", "confidence", "relative_depth", "proximity"]].head(20))


## 6. Match depth objects to tracking diagnostics

The depth artifact does not necessarily include bounding-box-to-track-ID associations. Therefore, this engine uses frame-level tracking persistence as contextual evidence and does **not** claim to identify the track of an individual depth object.


In [ ]:
tracking_lookup = {}

for row in tracking_df.itertuples(index=False):
    key = (
        str(getattr(row, "mode", "original")),
        str(getattr(row, "image_name", "")),
    )

    ids = parse_track_ids(getattr(row, "track_ids", ""))

    lifespans = []
    sequence_id = str(getattr(row, "sequence_id", ""))

    for tid in ids:
        lifespans.append(
            track_occurrences.get(
                (str(getattr(row, "mode", "original")), sequence_id, tid),
                1
            )
        )

    tracking_lookup[key] = {
        "active_tracks": int(getattr(row, "active_tracks", 0)),
        "detections": int(getattr(row, "detections", 0)),
        "mean_confidence": float(
            getattr(row, "mean_confidence", 0.0)
        ),
        "mean_track_lifespan": (
            float(np.mean(lifespans)) if lifespans else 0.0
        ),
    }

print("Tracking frame records:", len(tracking_lookup))


## 7. Calculate object-level risk contributions

Formula:

`object_risk = proximity × confidence_factor × (0.5 + 0.5 × persistence_factor) × class_weight`

The persistence term is based on the mean lifespan of tracks present in the frame, so it is contextual rather than object-specific. The scene-condition factor is applied after aggregation.


In [ ]:
def confidence_factor(confidence):
    c = float(np.clip(pd.to_numeric(confidence, errors="coerce") if pd.notna(confidence) else 0.0, 0, 1))
    floor = float(np.clip(confidence_floor, 0, 0.999999))
    return float(np.clip((c - floor) / (1 - floor), 0, 1))


def persistence_factor(lifespan):
    reference = max(int(persistence_reference_frames), 1)
    return float(np.clip(math.log1p(max(float(lifespan), 0)) / math.log1p(reference), 0, 1))


def object_risk(row, lifespan):
    proximity = float(np.clip(row["proximity"], 0, 1))
    conf = confidence_factor(row["confidence"])
    persistence = persistence_factor(lifespan)
    cls_weight = float(class_weight.get(int(row["class_id"]), 1.0))
    return proximity * conf * (0.5 + 0.5 * persistence) * max(cls_weight, 0.0)

risk_rows = []
for row in depth_df.itertuples(index=False):
    image_name = str(row.image_name)
    condition = str(row.condition) if pd.notna(row.condition) else "Unknown"
    cls = int(row.class_id)
    lookup = tracking_lookup.get(("original", image_name), {
        "active_tracks": 0, "detections": 0, "mean_confidence": 0.0, "mean_track_lifespan": 0.0
    })
    lifespan = lookup["mean_track_lifespan"]
    contribution = object_risk({"proximity": row.proximity, "confidence": row.confidence, "class_id": cls}, lifespan)
    relative_depth = float(row.relative_depth) if pd.notna(row.relative_depth) else np.nan
    risk_rows.append({
        "image_id": getattr(row, "image_id", image_name),
        "image_name": image_name,
        "condition": condition,
        "class_id": cls,
        "confidence": float(row.confidence),
        "relative_depth": relative_depth,
        "proximity": float(row.proximity),
        "active_tracks": lookup["active_tracks"],
        "frame_mean_confidence": lookup["mean_confidence"],
        "mean_track_lifespan": lifespan,
        "class_weight": float(class_weight.get(cls, 1.0)),
        "object_risk_contribution": contribution,
    })

object_risk_df = pd.DataFrame(risk_rows)
if object_risk_df.empty:
    object_risk_df = pd.DataFrame(columns=["image_id", "image_name", "condition", "class_id", "confidence", "relative_depth", "proximity", "active_tracks", "frame_mean_confidence", "mean_track_lifespan", "class_weight", "object_risk_contribution"])
display(object_risk_df.head(20))


## 8. Aggregate to frame-level risk

Multiple nearby objects can increase risk, but the aggregate is capped at 1.0 before applying the condition factor.


In [ ]:
def risk_level(score):
    if score >= level_thresholds["HIGH"]:
        return "HIGH"
    if score >= level_thresholds["MEDIUM"]:
        return "MEDIUM"
    if score >= level_thresholds["LOW"]:
        return "LOW"
    return "VERY_LOW"

frame_rows = []
for image_name, group in object_risk_df.groupby("image_name", sort=False):
    condition = str(group["condition"].iloc[0])
    raw_object_risk = float(group["object_risk_contribution"].sum())
    aggregate_object_risk = 1 - math.exp(-max(raw_object_risk, 0.0))
    condition_mult = float(condition_factor.get(condition, condition_factor.get("Unknown", 1.0)))
    scene_risk = float(np.clip(aggregate_object_risk * max(condition_mult, 0.0), 0, 1))
    frame_rows.append({
        "image_name": image_name, "condition": condition, "objects": int(len(group)),
        "max_proximity": float(group["proximity"].max()),
        "mean_proximity": float(group["proximity"].mean()),
        "max_object_risk": float(group["object_risk_contribution"].max()),
        "raw_object_risk": raw_object_risk, "aggregate_object_risk": aggregate_object_risk,
        "condition_factor": condition_mult, "risk_score": scene_risk, "risk_level": risk_level(scene_risk),
    })
frame_risk_df = pd.DataFrame(frame_rows)
if frame_risk_df.empty:
    frame_risk_df = pd.DataFrame(columns=["image_name", "condition", "objects", "max_proximity", "mean_proximity", "max_object_risk", "raw_object_risk", "aggregate_object_risk", "condition_factor", "risk_score", "risk_level"])
display(frame_risk_df.head(20))


## 9. Risk distribution and condition analysis

In [ ]:
if frame_risk_df.empty:
    risk_distribution = pd.DataFrame(columns=["risk_level", "frames"])
    condition_summary = pd.DataFrame(columns=["condition", "frames", "mean_risk", "median_risk", "max_risk"])
else:
    risk_distribution = frame_risk_df["risk_level"].value_counts().rename_axis("risk_level").reset_index(name="frames")
    condition_summary = (frame_risk_df.groupby("condition").agg(
        frames=("image_name", "count"), mean_risk=("risk_score", "mean"),
        median_risk=("risk_score", "median"), max_risk=("risk_score", "max")
    ).reset_index())

display(risk_distribution)
display(condition_summary)


In [ ]:
if not frame_risk_df.empty:
    plt.figure(figsize=(9, 5))
    plt.hist(frame_risk_df["risk_score"], bins=20)
    plt.xlabel("Risk score")
    plt.ylabel("Frames")
    plt.title("Risk Score Distribution")
    plt.tight_layout()
    plt.show()

if not condition_summary.empty:
    plt.figure(figsize=(9, 5))
    plt.bar(condition_summary["condition"], condition_summary["mean_risk"])
    plt.ylabel("Mean risk score")
    plt.title("Mean Risk by Road Condition")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 10. Highest-risk frames

These are the frames that the rule-based engine considers most urgent under the configured project scoring scheme.


In [ ]:
top_risk = frame_risk_df.sort_values("risk_score", ascending=False).head(30)
display(top_risk)


## 11. Save risk-engine artifacts

In [ ]:
OBJECT_PATH = RESULT_ROOT / "risk_object_contributions.csv"
FRAME_PATH = RESULT_ROOT / "risk_frame_scores.csv"
CONDITION_PATH = RESULT_ROOT / "risk_condition_summary.csv"
LEVEL_PATH = RESULT_ROOT / "risk_level_distribution.csv"
REPORT_PATH = RESULT_ROOT / "risk_engine_report.json"

object_risk_df.to_csv(OBJECT_PATH, index=False)
frame_risk_df.to_csv(FRAME_PATH, index=False)
condition_summary.to_csv(CONDITION_PATH, index=False)
risk_distribution.to_csv(LEVEL_PATH, index=False)

report = {
    "selected_detector": selection.get("selected_detector"),
    "selected_model": selection.get("selected_model"),
    "demo_data_used": bool(USE_DEMO_DATA and bool(missing)),
    "inputs": {
        "depth_file": str(DEPTH_PATH),
        "tracking_file": str(TRACK_PATH),
        "selection_file": str(SELECTION_PATH),
    },
    "configuration": {
        "near_distance_threshold": near_distance_threshold,
        "critical_distance_threshold": critical_distance_threshold,
        "confidence_floor": confidence_floor,
        "persistence_reference_frames": persistence_reference_frames,
        "condition_factor": condition_factor,
        "class_weight": class_weight,
        "level_thresholds": level_thresholds,
    },
    "risk_definition": (
        "Object contribution = proximity × confidence factor × "
        "persistence factor × class weight. Frame object contributions "
        "are saturating-aggregated and multiplied by the scene condition factor."
    ),
    "frame_count": int(len(frame_risk_df)),
    "object_count": int(len(object_risk_df)),
    "risk_levels": risk_distribution.to_dict(orient="records"),
    "condition_summary": condition_summary.to_dict(orient="records"),
    "note": (
        ("SYNTHETIC DEMONSTRATION DATA USED; do not interpret as project evaluation. " if (USE_DEMO_DATA and bool(missing)) else "") +
        "This is an interpretable perception-risk score for the project. "
        "It is not a collision probability, calibrated hazard probability, "
        "or safety certification."
    ),
}

REPORT_PATH.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("Saved:", OBJECT_PATH)
print("Saved:", FRAME_PATH)
print("Saved:", CONDITION_PATH)
print("Saved:", LEVEL_PATH)
print("Saved:", REPORT_PATH)


## Final output

Main artifacts:

```text
results/risk/risk_object_contributions.csv
results/risk/risk_frame_scores.csv
results/risk/risk_condition_summary.csv
results/risk/risk_level_distribution.csv
results/risk/risk_engine_report.json
```

Before interpreting results, confirm that the depth and tracking inputs are real outputs from the matching evaluation split. Synthetic demo rows are only for smoke-testing formulas and must not be reported as evaluation results.
